# 02 -- Feature Engineering

Sentinel handling (-1 -> NaN + `_is_missing` flag) and the engineered
features, each tied to one of three account-opening fraud archetypes:
(A) synthetic identity, (B) identity theft, (C) mule farming.
See `src/feature_engineering.py` and `src/preprocessing.py` for the
production code this notebook demonstrates.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config

cfg = load_config(ROOT / "config.yaml")
pd.set_option("display.max_columns", 40)


In [2]:
from src.data_loader import load_raw
from src.preprocessing import to_nan_and_flag, drop_constant_columns
from src.feature_engineering import add_features, SKIPPED_FEATURES

df = load_raw(cfg)
const = drop_constant_columns(df, cfg.data.target_col, keep=[cfg.data.month_col])
df = df.drop(columns=const)
print("dropped constant columns:", const)

dropped constant columns: ['device_fraud_count']


## Step 1 -- sentinel -1 -> NaN + `_is_missing` flag

Missingness is itself predictive here: a synthetic identity has no previous address precisely because it was invented last week. This step is also a correctness prerequisite -- without it, -1 silently corrupts every ratio built from these columns below.

In [3]:
before = df[cfg.sentinel_cols].describe()
df2 = to_nan_and_flag(df, cfg.sentinel_cols)
after = df2[cfg.sentinel_cols].describe()
print("BEFORE (raw, sentinel -1 mixed in):")
display(before)
print("\nAFTER (sentinel converted to NaN):")
display(after)
[c for c in df2.columns if c.endswith("_is_missing")]

BEFORE (raw, sentinel -1 mixed in):


,prev_address_months_count,current_address_months_count,bank_months_count,session_length_in_minutes,device_distinct_emails_8w,intended_balcon_amount
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,16.718568,86.587867,10.839303,7.544940,1.018312,8.661499
std,44.046230,88.406599,12.116875,8.033106,0.180761,20.236155
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-15.530555
25%,-1.000000,19.000000,-1.000000,3.103053,1.000000,-1.181488
50%,-1.000000,52.000000,5.000000,5.114321,1.000000,-0.830507
75%,12.000000,130.000000,25.000000,8.866131,1.000000,4.984176
max,383.000000,428.000000,32.000000,85.899143,2.000000,112.956928



AFTER (sentinel converted to NaN):


,prev_address_months_count,current_address_months_count,bank_months_count,session_length_in_minutes,device_distinct_emails_8w,intended_balcon_amount
count,287080.000000,995746.000000,746365.000000,997985.000000,999641.000000,257477.000000
mean,60.719967,86.962058,14.862618,7.562193,1.019037,36.582496
std,63.578187,88.409289,11.527847,8.032021,0.176700,23.236885
min,5.000000,0.000000,1.000000,0.000872,0.000000,0.000054
25%,25.000000,20.000000,1.000000,3.117642,1.000000,20.403236
50%,34.000000,53.000000,15.000000,5.122832,1.000000,32.433701
75%,72.000000,130.000000,28.000000,8.878215,1.000000,49.586253
max,383.000000,428.000000,32.000000,85.899143,2.000000,112.956928


['prev_address_months_count_is_missing',
 'current_address_months_count_is_missing',
 'bank_months_count_is_missing',
 'session_length_in_minutes_is_missing',
 'device_distinct_emails_8w_is_missing',
 'intended_balcon_amount_is_missing']

## Step 2 -- engineered features

In [4]:
df3 = add_features(df2)
new_cols = [c for c in df3.columns if c not in df2.columns]
print(f"{len(new_cols)} engineered features added:")
for c in new_cols:
    print(" -", c)

16 engineered features added:
 - velocity_burst_6h_4w
 - velocity_ratio_6h_24h
 - velocity_burst_24h_4w
 - email_mismatch_free
 - dob_emails_x_mismatch
 - total_address_history
 - thin_file_score
 - n_missing
 - n_valid_phones
 - no_valid_phone
 - limit_to_income
 - limit_per_risk
 - risk_x_income
 - emails_per_session_min
 - short_session_no_keepalive
 - zip_density_vs_velocity


In [5]:
df3[new_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
velocity_burst_6h_4w,1000000.0,1.160970,0.571274,0.000000,0.755319,1.131597,1.513168,5.287729
velocity_ratio_6h_24h,1000000.0,1.209314,0.577612,0.000000,0.806072,1.215435,1.558393,8.620877
velocity_burst_24h_4w,1000000.0,0.986199,0.268216,0.232317,0.791944,0.956378,1.154253,3.003246
email_mismatch_free,1000000.0,0.277938,0.335381,0.000000,0.000000,0.095870,0.560289,0.999999
dob_emails_x_mismatch,1000000.0,4.778235,4.001045,0.000000,1.683395,3.716734,6.869775,36.339294
total_address_history,1000000.0,104.023609,87.473062,0.000000,40.000000,69.000000,145.000000,720.000000
thin_file_score,1000000.0,0.966555,0.601550,0.000000,1.000000,1.000000,1.000000,2.000000
n_missing,1000000.0,1.715706,0.815931,0.000000,1.000000,2.000000,2.000000,5.000000
n_valid_phones,1000000.0,1.306753,0.507070,0.000000,1.000000,1.000000,2.000000,2.000000
no_valid_phone,1000000.0,0.022232,0.147437,0.000000,0.000000,0.000000,0.000000,1.000000


## Correlation of engineered features with the target (quick sanity check)

In [6]:
corrs = df3[new_cols + [cfg.data.target_col]].corr(numeric_only=True)[cfg.data.target_col].drop(cfg.data.target_col)
corrs.sort_values(key=abs, ascending=False)

risk_x_income                 0.082932
n_missing                     0.060254
thin_file_score               0.057523
limit_per_risk                0.052506
n_valid_phones               -0.042301
email_mismatch_free           0.041114
short_session_no_keepalive    0.030095
no_valid_phone                0.024276
total_address_history         0.021198
velocity_burst_6h_4w         -0.014039
velocity_ratio_6h_24h        -0.012225
zip_density_vs_velocity       0.010366
limit_to_income               0.009331
dob_emails_x_mismatch        -0.007532
emails_per_session_min        0.004051
velocity_burst_24h_4w        -0.003768
Name: fraud_bool, dtype: float64

## What was explicitly skipped, and why

No transaction amount / timestamp / history exists in this dataset, so `amount_log`, `transactions_per_hour`, `hour`/`day_of_week`/`is_weekend`, and the `current_amount / historical_average` deviation feature were skipped rather than faked with substitute columns.

In [7]:
for name, reason in SKIPPED_FEATURES.items():
    print(f"{name:40s} SKIPPED -- {reason}")

amount_log                               SKIPPED -- no transaction amount column exists in account-opening data
transactions_per_hour                    SKIPPED -- no transaction history exists; only application-time features
hour                                     SKIPPED -- no real timestamp, only a coarse 0-7 month index
day_of_week                              SKIPPED -- no real timestamp, only a coarse 0-7 month index
is_weekend                               SKIPPED -- no real timestamp, only a coarse 0-7 month index
current_amount_vs_historical_average     SKIPPED -- no transaction history to compute a historical average from
